**Examine both Open Meteo and Tomorrow and decide which one you'd prefer to use. What drove your decision?**

I decided to use Open-Meteo instead of Tomorrow.io.

Both APIs provide similar weather information, such as current temperature, apparent or “feels like” temperature, historical temperature with dates. 

Tomorrow.io has one advantage: it accepts more flexible location inputs, such as city names, postal codes, latitude/longitude, and possibly airport-style searches. 

But tomorrow.io requires an API key and appears to have usage limits.
Open-Meteo felt more straightforward to me. Its documentation was much clearer to me, it does not require an API key, and the API URL is easy to understand. 

The main limitation is that the open meteo’s API requires geographic coordinates instead of plain text location names. However, since we learned geocoding, I can work around this by first converting a city or airport location into latitude and longitude, and then using those coordinates in the weather request.

So I chose Open-Meteo because it is free, does not need an API key, has clear documentation, and works well once I add a geocoding step for locations.

**What is the URL to the documentation? (You don't use code for this one)**

Here is the [documentation](https://open-meteo.com/en/docs)

**Make a request for the current weather where you are born, or somewhere you've lived.**

In [24]:
#Note that I also did $ pip install openmeteo-requests according to open meteo documentation
import requests
from pprint import pprint
import geocoder
from dotenv import load_dotenv
load_dotenv()
import os
# I tried to do this with openmeteo-requests but it didn't really work!

***Setting up geocoder to get lat - long for all locations***

In [13]:
api_key = os.getenv('api_key_geocoding')

In [39]:
#Finding latitude and longitude for my location
g = geocoder.google('New Delhi', key=api_key)
lat_long_ND = g.latlng
lat_long_ND

[28.6139298, 77.2088282]

In [29]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 28.6214,
	"longitude": 77.2148,
	"current": ["temperature_2m", "precipitation", "rain", "showers", "snowfall", "relative_humidity_2m", "apparent_temperature", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m"]
}
responses = requests.get(url, params = params)
pprint(responses.json())
# I don't like that I have to specify everything to get full weather information. I do like the output format, with units specified in the output itself.

{'current': {'apparent_temperature': 41.6,
             'interval': 900,
             'precipitation': 0.0,
             'rain': 0.0,
             'relative_humidity_2m': 22,
             'showers': 0.0,
             'snowfall': 0.0,
             'temperature_2m': 41.7,
             'time': '2026-06-10T11:45',
             'wind_direction_10m': 270,
             'wind_gusts_10m': 34.2,
             'wind_speed_10m': 13.3},
 'current_units': {'apparent_temperature': '°C',
                   'interval': 'seconds',
                   'precipitation': 'mm',
                   'rain': 'mm',
                   'relative_humidity_2m': '%',
                   'showers': 'mm',
                   'snowfall': 'cm',
                   'temperature_2m': '°C',
                   'time': 'iso8601',
                   'wind_direction_10m': '°',
                   'wind_gusts_10m': 'km/h',
                   'wind_speed_10m': 'km/h'},
 'elevation': 217.0,
 'generationtime_ms': 0.19061565399169922,
 'la

**Print out the country this location is in.**

In [48]:
g_reverse = geocoder.google(lat_long_ND, key = api_key, method='reverse') # from geocoder documentation 
pprint(g_reverse.country_long) # I initially did .country but got IN only, then went back to docs and found you can do country_long to get full name

'India'


In [ ]:
#I had to use geocoder to get location whereas weather API had all of this together

**Print out the difference between the current temperature and how warm it feels. Use "It feels ___ degrees colder" or "It feels ___ degrees warmer," not negative numbers.**

In [52]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 28.6214,
	"longitude": 77.2148,
	"current": ["temperature_2m", "apparent_temperature"]
}
responses = requests.get(url, params = params)
data = responses.json()
pprint(data)

{'current': {'apparent_temperature': 41.4,
             'interval': 900,
             'temperature_2m': 41.3,
             'time': '2026-06-10T12:00'},
 'current_units': {'apparent_temperature': '°C',
                   'interval': 'seconds',
                   'temperature_2m': '°C',
                   'time': 'iso8601'},
 'elevation': 217.0,
 'generationtime_ms': 455.8354616165161,
 'latitude': 28.646748,
 'longitude': 77.17218,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'utc_offset_seconds': 0}


In [54]:
if data['current']['apparent_temperature'] > data['current']['temperature_2m']:
    print(f"It feels {(data['current']['apparent_temperature'] - data['current']['temperature_2m']):.2f} degrees warmer")
else:
    print(f"It feels {(data['current']['temperature_2m'] - data['current']['apparent_temperature']):.2f} degrees colder")

It feels 0.10 degrees warmer


**What's the current temperature at Heathrow International Airport? Use the airport's IATA code to search.**

In [56]:
g_airport = geocoder.google('Heathrow International Airport', key=api_key)
lat_long_airport = g_airport.latlng
lat_long_airport

[51.4679903, -0.4550471]

In [59]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": lat_long_airport[0],
	"longitude": lat_long_airport[1],
	"current": ["temperature_2m"]}
    
responses = requests.get(url, params = params)
pprint(responses.json())
# the current temperature is 15.8 degrees celsius

{'current': {'interval': 900,
             'temperature_2m': 15.8,
             'time': '2026-06-10T12:00'},
 'current_units': {'interval': 'seconds',
                   'temperature_2m': '°C',
                   'time': 'iso8601'},
 'elevation': 23.0,
 'generationtime_ms': 0.05924701690673828,
 'latitude': 51.5,
 'longitude': -0.5,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'utc_offset_seconds': 0}


**What URL would I use to request a 3-day forecast at Heathrow?**

In [76]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat_long_airport[0],
	"longitude": lat_long_airport[1],
	"forecast_days": 3,
    "daily": ["temperature_2m_min", "temperature_2m_max", "weather_code", "apparent_temperature_max", "apparent_temperature_min", "showers_sum", "rain_sum", "precipitation_sum", "sunrise", "sunset", "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant"]
}

responses = requests.get(url, params = params)
data_3dayforecast = responses.json()
pprint(data_3dayforecast)

{'daily': {'apparent_temperature_max': [14.2, 14.2, 19.0],
           'apparent_temperature_min': [6.9, 9.2, 12.9],
           'precipitation_sum': [2.5, 7.5, 0.13],
           'rain_sum': [2.5, 7.5, 0.13],
           'showers_sum': [3.3, 0.0, 0.1],
           'sunrise': ['2026-06-10T03:45',
                       '2026-06-11T03:44',
                       '2026-06-12T03:44'],
           'sunset': ['2026-06-10T20:17',
                      '2026-06-11T20:18',
                      '2026-06-12T20:18'],
           'temperature_2m_max': [16.9, 16.7, 21.9],
           'temperature_2m_min': [9.4, 10.7, 15.2],
           'time': ['2026-06-10', '2026-06-11', '2026-06-12'],
           'weather_code': [55, 61, 3],
           'wind_direction_10m_dominant': [254, 238, 261],
           'wind_gusts_10m_max': [34.6, 48.2, 55.4],
           'wind_speed_10m_max': [17.6, 27.4, 27.0]},
 'daily_units': {'apparent_temperature_max': '°C',
                 'apparent_temperature_min': '°C',
                 

**Print the date of each of the 3 days you're getting a forecast for.**

In [71]:
data_3dayforecast['daily']['time']

['2026-06-10', '2026-06-11', '2026-06-12']

**Print the maximum temperature of each of the days.**

In [72]:
data_3dayforecast['daily']['temperature_2m_max']
# this was much easier to do here

[16.9, 16.7, 21.9]

**Print only the day with the highest maximum temperature.**

In [74]:
forecast_date_info = [
    {"date": data_3dayforecast['daily']['time'][0],
    "maxtemp": data_3dayforecast['daily']['temperature_2m_max'][0]},
    {"date": data_3dayforecast['daily']['time'][1],
     "maxtemp": data_3dayforecast['daily']['temperature_2m_max'][1]},
    {"date": data_3dayforecast['daily']['time'][2],
     "maxtemp": data_3dayforecast['daily']['temperature_2m_max'][2]}]

forecast_date_info

[{'date': '2026-06-10', 'maxtemp': 16.9},
 {'date': '2026-06-11', 'maxtemp': 16.7},
 {'date': '2026-06-12', 'maxtemp': 21.9}]

In [75]:
max_temp=0

for i in forecast_date_info:
    if i["maxtemp"]>max_temp:
        max_temp = i["maxtemp"]
        maxtemp_forecastdate = i["date"]
print(f"The maximum temperature was {max_temp} on the date {maxtemp_forecastdate}")

The maximum temperature was 21.9 on the date 2026-06-12


**Did you find this easier or more difficult than using the weatherapi.com, and why? Which would you recommend to someone interesting in building a tool around weather information?**

I found WeatherAPI easier to use than Open-Meteo. Its documentation was cleaner and more intuitive, and it did not require an additional geocoding step to convert place names into latitude and longitude coordinates. WeatherAPI also returned useful information such as country names directly in the response, whereas with Open-Meteo I had to use reverse geocoding to obtain that information.
One advantage of Open-Meteo was that it made it straightforward to request specific forecast data, such as daily maximum temperatures. However, I generally found WeatherAPI more convenient because it returned a richer set of weather information by default, without requiring me to explicitly specify as many parameters.